# Первый день минутных свечей BTC из S3

Notebook находит самый ранний дневной parquet для `BTCUSDT/1m` и загружает его в `df`. Доступ к S3 берётся из переменных окружения `YC_ENDPOINT`, `YC_REGION`, `YC_ACCESS_KEY_ID`, `YC_SECRET_ACCESS_KEY`; бакет можно переопределить через `YC_BUCKET`.

In [ ]:
import io
import os

import boto3
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

required_env = [
    "YC_ENDPOINT",
    "YC_REGION",
    "YC_ACCESS_KEY_ID",
    "YC_SECRET_ACCESS_KEY",
]
missing_env = [name for name in required_env if not os.getenv(name)]
if missing_env:
    raise RuntimeError(
        "Не настроен доступ к S3. Добавьте в .env: "
        + ", ".join(missing_env)
    )

BUCKET = os.getenv("YC_BUCKET", "binance-data-downloader")
PREFIX = "raw/klines/symbol=BTCUSDT/interval=1m/"

s3 = boto3.client(
    "s3",
    endpoint_url=os.getenv("YC_ENDPOINT"),
    region_name=os.getenv("YC_REGION"),
    aws_access_key_id=os.getenv("YC_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("YC_SECRET_ACCESS_KEY"),
)

In [ ]:
response = s3.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX, MaxKeys=1)
objects = response.get("Contents", [])
if not objects:
    raise FileNotFoundError(f"В s3://{BUCKET}/{PREFIX} не найдены свечи")

first_key = objects[0]["Key"]
first_key

In [ ]:
body = s3.get_object(Bucket=BUCKET, Key=first_key)["Body"].read()
df = pd.read_parquet(io.BytesIO(body))

print(f"Источник: s3://{BUCKET}/{first_key}")
print(f"Размер: {df.shape}")
df.head()

In [ ]:
assert len(df) == 1440, f"Ожидалось 1440 минут, получено {len(df)}"
assert df["timestamp"].dt.strftime("%Y-%m-%d").nunique() == 1

df.info()
df